# TKG Paper

In [1]:
import pandas as pd
import os
import time

First tests

In [2]:
promptfest="List only the 100 most likely festivals, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information—only this ranked list should be the answer."

promptart="List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information—only this ranked list should be the answer."


In [ ]:

# Read files
with open("Queries_fuer_LLMs/artist_test.txt", "r", encoding="utf-8") as f:
    artist_lines = f.read().splitlines()

with open("Queries_fuer_LLMs/festival_location_test.txt", "r", encoding="utf-8") as f:
    festival_lines = f.read().splitlines()

festival_data = []
for line in festival_lines:
    if line.startswith("FESTIVAL:"):
        content = line.replace("FESTIVAL:", "")
        
        content_clean = content.replace("_", "")
        parts = content_clean.split(";")
        
        name = parts[0] if len(parts) > 0 else ""
        country = parts[1] if len(parts) > 1 else ""
        city = parts[2] if len(parts) > 2 else ""
        
        question = f'What artists will perform at the 2025 {name} in {city}, {country}?'
        
        performs_at_festival = f'?,performs_at_festival,{name}. The festival happens in {country};{city}.'
        
        festival_data.append({
            "TKG_query": performs_at_festival,
            "type": "festival",
            "col1": name,
            "col2": country,
            "col3": city,
            "full_string": ";".join(parts),
            "full_stringOG": line,
            "prompt": f'{question} {promptart}'
        })
artist_data = []
for line in artist_lines:
    if line.startswith("ARTIST:"):
        content = line.replace("ARTIST:", "")
        
        content_clean = content.replace("_", " ").strip()

        parts = content_clean.split(";")
        
        name = parts[0] if len(parts) > 0 else ""
        
        question = f'At what festivals will {name} perform in 2025?'
        
        performs_at_festival = f'{name},performs_at_festival,?,2025'
        
        artist_data.append({
            "TKG_query": performs_at_festival,
            "type": "artist",
            "col1": name,
            "full_string": ";".join(parts),
            "full_stringOG": line,
            "prompt": f'{question} {promptfest}'
        })

df = pd.DataFrame(festival_data + artist_data)

df.to_csv("prompt_output.csv", index=False)

## OPENSOURCE

In [ ]:
from openai import OpenAI
keyLINK = "KEY"
keyVLLM = "KEY"

client = OpenAI(
     base_url=keyLINK,
     api_key=keyVLLM
     
)

### 70B

In [ ]:
import pandas as pd
import time
import csv
import os

df = pd.DataFrame(festival_data + artist_data)

file_path = "prompt_output_Meta-Llama-3.3-70B.csv"
file_exists = os.path.isfile(file_path)

with open(file_path, mode="a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    
    if not file_exists:
        writer.writerow(df.columns.tolist() + ["answer"])
    
    for index, row in df.iterrows():
        question_text = row['prompt']
        
        response = client.chat.completions.create(
            model="Meta-Llama-3.3-70B-Instruct-AWQ-INT4",
            messages=[{"role": "user", "content": question_text}],
            temperature=0,
        )
        
        answer_text = response.choices[0].message.content.strip()
        
        writer.writerow(list(row) + [answer_text])
        
        df.at[index, 'answer'] = answer_text
      


## 8B

In [ ]:
import pandas as pd
import time
import csv
import os

df = pd.DataFrame(festival_data + artist_data)

file_path = "prompt_output_Meta-Llama-3-8B.csv"
file_exists = os.path.isfile(file_path)

with open(file_path, mode="a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    
    if not file_exists:
        writer.writerow(df.columns.tolist() + ["answer"])
    
    for index, row in df.iterrows():
        question_text = row['prompt']
        
        response = client.chat.completions.create(
            model="llama-3.1-8b",
            messages=[{"role": "user", "content": question_text}],
            temperature=0,
        )
        
        answer_text = response.choices[0].message.content.strip()
        
        writer.writerow(list(row) + [answer_text])
        
        df.at[index, 'answer'] = answer_text
      


## Mixtral

In [ ]:
import pandas as pd
import time
import csv
import os

df = pd.DataFrame(festival_data + artist_data)

file_path = "prompt_output_mixtral-8x7b.csv"
file_exists = os.path.isfile(file_path)

with open(file_path, mode="a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    
    if not file_exists:
        writer.writerow(df.columns.tolist() + ["answer"])
    
    for index, row in df.iterrows():
        question_text = row['prompt']
        
        response = client.chat.completions.create(
            model="mixtral-8x7b",
            messages=[{"role": "user", "content": question_text}],
            temperature=0,
        )
        
        answer_text = response.choices[0].message.content.strip()
        
        writer.writerow(list(row) + [answer_text])
        
        df.at[index, 'answer'] = answer_text
      

# Test prompts

In [9]:
lista=["What artists will perform at the 2025 Bautz Festival in Lüdenscheid, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer.",
       "What artists will perform at the 2025 Giessener Kultursommer in Giessen, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer.",
       "What artists will perform at the 2025 Music Forge Festival in Lich, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer.",
       
       "At what festivals will ASP perform in 2025? List only the 100 most likely festivals, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer.",
       "At what festivals will Vogon Poetry  perform in 2025? List only the 100 most likely festivals, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer."
       ]

In [10]:
listb=["What artists will perform at the 2025 Bautz Festival in Lüdenscheid, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       "What artists will perform at the 2025 Giessener Kultursommer in Giessen, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       "What artists will perform at the 2025 Music Forge Festival in Lich, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       
       "At what festivals in Germany does ASP perform in 2025? List only the 100 most likely festivals, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       "At what festivals in Germany does Vogon Poetry perform in 2025? List only the 100 most likely festivals, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely."
]



In [11]:
listc=["What artists are likely to perform at the 2025 Bautz Festival in Lüdenscheid, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       "What artists are likely to perform at the 2025 Giessener Kultursommer in Giessen, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       "What artists are likely to perform at the 2025 Music Forge Festival in Lich, Germany? List only the 100 most likely artists, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       
       "At which festivals in Germany is ASP likely to perform in 2025? List only the 100 most likely festivals, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely.",
       "At which festivals in Germany is Vogon Poetry likely to perform in 2025? List only the 100 most likely festivals, sorted by likelihood from 1 (most likely) to 100 (least likely). Do not provide any other information, only this ranked list should be the answer. Even if you are not sure, make a list of the 100 you consider most likely."
       ]

In [13]:
listd=["""
You are estimating likely performers for the Bautz Festival 2025 in Lüdenscheid, Germany.
Task:
Generate a ranked list of 100 artists most likely to perform.
Output requirements:
- Provide exactly 100 artist names
- Rank them from 1 (most likely) to 100 (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Artist Name", "2. Artist Name", etc.
- Do NOT include explanations, notes, or extra text
If uncertain, still provide your best estimate based on the criteria above.
""",

"""
You are estimating likely performers for the Giessener Kultursommer 2025 in Giessen, Germany.
Task:
Generate a ranked list of 100 artists most likely to perform.
Output requirements:
- Provide exactly 100 artist names
- Rank them from 1 (most likely) to 100 (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Artist Name", "2. Artist Name", etc.
- Do NOT include explanations, notes, or extra text
If uncertain, still provide your best estimate based on the criteria above.
""",

"""
You are estimating likely performers for the Music Forge Festival 2025 in Lich, Germany.
Task:
Generate a ranked list of 100 artists most likely to perform.
Output requirements:
- Provide exactly 100 artist names
- Rank them from 1 (most likely) to 100 (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Artist Name", "2. Artist Name", etc.
- Do NOT include explanations, notes, or extra text
If uncertain, still provide your best estimate based on the criteria above.
""",




"""You are estimating which festivals in Germany an artist named "ASP" is most likely to perform at in 2025.
Task:
Generate a ranked list of 100 German festivals where "ASP" is most likely to perform in 2025.
Output requirements:
- Provide exactly 100 festival names
- Rank them from 1 (most likely) to 100 (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Festival Name", "2. Festival Name", etc.
- Do NOT include explanations or any extra text
If uncertain, still provide your best estimate based on the criteria above.
""",

"""You are estimating which festivals in Germany an artist named "Vogon Poetry" is most likely to perform at in 2025.
Task:
Generate a ranked list of 100 German festivals where "Vogon Poetry" is most likely to perform in 2025.
Output requirements:
- Provide exactly 100 festival names
- Rank them from 1 (most likely) to 100 (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Festival Name", "2. Festival Name", etc.
- Do NOT include explanations or any extra text
If uncertain, still provide your best estimate based on the criteria above.
""",

]

In [ ]:
import pandas as pd
import csv
import os
import time



lists = [lista, listb, listc, listd]

# Models to use
models = ["mixtral-8x7b", "llama-3.1-8b", "Meta-Llama-3.3-70B-Instruct-AWQ-INT4"]
hardcoded_ids = ["PA", "PB", "PC", "PD"]

file_path = "llm_responses.csv"
file_exists = os.path.isfile(file_path)

with open(file_path, mode="a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    
    if not file_exists:
        writer.writerow(["list_name", "item", "model", "prompt_id", "answer"])
    
    for list_index, current_list in enumerate(lists):
        hardcoded_id = hardcoded_ids[list_index]
        list_name = f"list{list_index+1}"
        
        for item in current_list:
            for model_name in models:

                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": item}],
                    temperature=0,
                )
                
                answer_text = response.choices[0].message.content.strip()
                
                writer.writerow([list_name, item, model_name, hardcoded_id, answer_text])
                
                time.sleep(0.5)



# Time Tests

In [ ]:
import pandas as pd

# Read files
with open("OneDrive_1_26-03-2026/202320242025_artist_test_subset.txt", "r", encoding="utf-8") as f:
    artist_lines = f.read().splitlines()

with open("OneDrive_1_26-03-2026/202320242025_festival_location_test_subset.txt", "r", encoding="utf-8") as f:
    festival_lines = f.read().splitlines()

def process_festival_lines(lines, year, n_festival):
    data = []
    for line in lines:
        if ";" in line:
            id_part, content_part = line.split(";", 1)
        else:
            id_part, content_part = "", line

        if content_part.startswith("FESTIVAL:"):
            content = content_part.replace("FESTIVAL:", "")
            content_clean = content.replace("_", "")
            parts = content_clean.split(";")

            name = parts[0] if len(parts) > 0 else ""
            country = parts[1] if len(parts) > 1 else ""
            city = parts[2] if len(parts) > 2 else ""

            question=f"""
You are estimating likely performers for the {name} {year} in {city}, {country}.
Task:
Generate a ranked list of {n_festival} artists most likely to perform.
Output requirements:
- Provide exactly {n_festival} artist names
- Rank them from 1 (most likely) to {n_festival} (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Artist Name", "2. Artist Name", etc.
- Do NOT include explanations, notes, or extra text
If uncertain, still provide your best estimate based on the criteria above.
"""
            performs_at_festival = f'?,performs_at_festival,{name}. The festival happens in {country};{city}.'

            data.append({
                "id": id_part,
                "TKG_query": performs_at_festival,
                "type": "festival",
                "col1": name,
                "col2": country,
                "col3": city,
                "full_string": ";".join(parts),
                "full_stringOG": content_part,
                "prompt": f'{question}',
                "n_response": n_festival
            })
    return data

def process_artist_lines(lines, year, n_artist):
    data = []
    for line in lines:
        if ";" in line:
            id_part, content_part = line.split(";", 1)
        else:
            id_part, content_part = "", line

        if content_part.startswith("ARTIST:"):
            content = content_part.replace("ARTIST:", "")
            content_clean = content.replace("_", " ").strip()
            parts = content_clean.split(";")

            name = parts[0] if len(parts) > 0 else ""
            question = f"""
You are estimating which festivals in Germany an artist named "{name}" is most likely to perform at in {year}.
Task:
Generate a ranked list of {n_artist} German festivals where "{name}" is most likely to perform in {year}.
Output requirements:
- Provide exactly {n_artist} festival names
- Rank them from 1 (most likely) to {n_artist} (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Festival Name", "2. Festival Name", etc.
- Do NOT include explanations or any extra text
If uncertain, still provide your best estimate based on the criteria above.
"""
            performs_at_festival = f'{name},performs_at_festival,?,{year}'

            data.append({
                "id": id_part,
                "TKG_query": performs_at_festival,
                "type": "artist",
                "col1": name,
                "full_string": ";".join(parts),
                "full_stringOG": content_part,
                "prompt": f'{question}',
                "n_response": n_artist
            })
    return data

# Define multiple sizes
artist_sizes = [10, 50]
festival_sizes = [10, 210]

for year in [2023, 2024, 2025]:
    festival_data_all = []
    for n_festival in festival_sizes:
        festival_data_all.extend(process_festival_lines(festival_lines, year, n_festival))

    artist_data_all = []
    for n_artist in artist_sizes:
        artist_data_all.extend(process_artist_lines(artist_lines, year, n_artist))

    df = pd.DataFrame(festival_data_all + artist_data_all)
    
    csv_name = f"results_time/prompt_output_{year}.csv"
    df.to_csv(csv_name, index=False)


## Llama 8B

In [ ]:
import pandas as pd
import time
import csv
import os

t2023 = pd.read_csv("results_time/prompt_output_2023.csv")
t2024 = pd.read_csv("results_time/prompt_output_2024.csv")
t2025 = pd.read_csv("results_time/prompt_output_2025.csv")

for dataset, name in zip([t2023,t2024,t2025], ["t2023","t2024","t2025"]):

    file_path = "results_time/prompt_output_Meta-Llama-3-8B"+str(name)+".csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        
        if not file_exists:
            writer.writerow(dataset.columns.tolist() + ["answer"])
        
        for index, row in dataset.iterrows():
            question_text = row['prompt']
            
            response = client.chat.completions.create(
                model="llama-3.1-8b",
                messages=[{"role": "user", "content": question_text}],
                temperature=0,
            )
            
            answer_text = response.choices[0].message.content.strip()
            
            writer.writerow(list(row) + [answer_text])
            
            dataset.at[index, 'answer'] = answer_text
        



output_files = [
    "results_time/prompt_output_Meta-Llama-3-8Bt2023.csv",
    "results_time/prompt_output_Meta-Llama-3-8Bt2024.csv",
    "results_time/prompt_output_Meta-Llama-3-8Bt2025.csv"
]

for file_path in output_files:
    df = pd.read_csv(file_path)
    
    df_selected = df[["id", "type","n_response","answer"]]
    
    new_file_path = file_path.replace(".csv", "_CM.csv")
    
    df_selected.to_csv(new_file_path, index=False)
    


## 70B

In [ ]:
import pandas as pd
import time
import csv
import os

t2023 = pd.read_csv("results_time/prompt_output_2023.csv")
t2024 = pd.read_csv("results_time/prompt_output_2024.csv")
t2025 = pd.read_csv("results_time/prompt_output_2025.csv")

for dataset, name in zip([t2023,t2024,t2025], ["t2023","t2024","t2025"]):

    file_path = "results_time/prompt_output_Meta-Llama-70B"+str(name)+".csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        
        if not file_exists:
            writer.writerow(dataset.columns.tolist() + ["answer"])
        
        for index, row in dataset.iterrows():
            question_text = row['prompt']
            
            response = client.chat.completions.create(
                model="Meta-Llama-3.3-70B-Instruct-AWQ-INT4",
                messages=[{"role": "user", "content": question_text}],
                temperature=0,
            )
            
            answer_text = response.choices[0].message.content.strip()
            
            writer.writerow(list(row) + [answer_text])
            
            dataset.at[index, 'answer'] = answer_text
        

output_files = [
    "results_time/prompt_output_Meta-Llama-70Bt2023.csv",
    "results_time/prompt_output_Meta-Llama-70Bt2024.csv",
    "results_time/prompt_output_Meta-Llama-70Bt2025.csv"
]

for file_path in output_files:
    df = pd.read_csv(file_path)
    
    df_selected = df[["id", "type", "n_response","answer"]]
    
    new_file_path = file_path.replace(".csv", "_CM.csv")
    
    df_selected.to_csv(new_file_path, index=False)
    


## Mixtral

In [ ]:
import pandas as pd
import time
import csv
import os

t2023 = pd.read_csv("results_time/prompt_output_2023.csv")
t2024 = pd.read_csv("results_time/prompt_output_2024.csv")
t2025 = pd.read_csv("results_time/prompt_output_2025.csv")

for dataset, name in zip([t2023,t2024,t2025], ["t2023","t2024","t2025"]):

    file_path = "results_time/prompt_output_mixtral-8x7b"+str(name)+".csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        
        if not file_exists:
            writer.writerow(dataset.columns.tolist() + ["answer"])
        
        for index, row in dataset.iterrows():
            question_text = row['prompt']
            
            response = client.chat.completions.create(
                model="mixtral-8x7b",
                messages=[{"role": "user", "content": question_text}],
                temperature=0,
            )
            
            answer_text = response.choices[0].message.content.strip()
            
            writer.writerow(list(row) + [answer_text])
            
            dataset.at[index, 'answer'] = answer_text
        


output_files = [
    "results_time/prompt_output_mixtral-8x7bt2023.csv",
    "results_time/prompt_output_mixtral-8x7bt2024.csv",
    "results_time/prompt_output_mixtral-8x7bt2025.csv"
]

for file_path in output_files:
    df = pd.read_csv(file_path)
    
    df_selected = df[["id", "type", "n_response","answer"]]
    
    new_file_path = file_path.replace(".csv", "_CM.csv")
    
    df_selected.to_csv(new_file_path, index=False)
    


# FINAL TEST

In [ ]:
import pandas as pd

with open("all_tests/alltest_artist_test.txt", "r", encoding="utf-8") as f:
    artist_lines = f.read().splitlines()

with open("all_tests/alltest_festival_location_test.txt", "r", encoding="utf-8") as f:
    festival_lines = f.read().splitlines()

def process_festival_lines(lines, year, n_festival):
    data = []
    for line in lines:
        if ";" in line:
            id_part, content_part = line.split(";", 1)
        else:
            id_part, content_part = "", line

        if content_part.startswith("FESTIVAL:"):
            content = content_part.replace("FESTIVAL:", "")
            content_clean = content.replace("_", "")
            parts = content_clean.split(";")

            name = parts[0] if len(parts) > 0 else ""
            country = parts[1] if len(parts) > 1 else ""
            city = parts[2] if len(parts) > 2 else ""

            question=f"""
You are estimating likely performers for the {name} {year} in {city}, {country}.
Task:
Generate a ranked list of {n_festival} artists most likely to perform.
Output requirements:
- Provide exactly {n_festival} artist names
- Rank them from 1 (most likely) to {n_festival} (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Artist Name", "2. Artist Name", etc.
- Do NOT include explanations, notes, or extra text
If uncertain, still provide your best estimate based on the criteria above.
"""
            performs_at_festival = f'?,performs_at_festival,{name}. The festival happens in {country};{city}.'

            data.append({
                "id": id_part,
                "TKG_query": performs_at_festival,
                "type": "festival",
                "col1": name,
                "col2": country,
                "col3": city,
                "full_string": ";".join(parts),
                "full_stringOG": content_part,
                "prompt": f'{question}',
                "n_response": n_festival
            })
    return data

def process_artist_lines(lines, year, n_artist):
    data = []
    for line in lines:
        if ";" in line:
            id_part, content_part = line.split(";", 1)
        else:
            id_part, content_part = "", line

        if content_part.startswith("ARTIST:"):
            content = content_part.replace("ARTIST:", "")
            content_clean = content.replace("_", " ").strip()
            parts = content_clean.split(";")

            name = parts[0] if len(parts) > 0 else ""
            question = f"""
You are estimating which festivals in Germany an artist named "{name}" is most likely to perform at in {year}.
Task:
Generate a ranked list of {n_artist} German festivals where "{name}" is most likely to perform in {year}.
Output requirements:
- Provide exactly {n_artist} festival names
- Rank them from 1 (most likely) to {n_artist} (least likely)
- Output ONLY the ranked list
- Format: each line should be "1. Festival Name", "2. Festival Name", etc.
- Do NOT include explanations or any extra text
If uncertain, still provide your best estimate based on the criteria above.
"""
            performs_at_festival = f'{name},performs_at_festival,?,{year}'

            data.append({
                "id": id_part,
                "TKG_query": performs_at_festival,
                "type": "artist",
                "col1": name,
                "full_string": ";".join(parts),
                "full_stringOG": content_part,
                "prompt": f'{question}',
                "n_response": n_artist
            })
    return data

artist_sizes = [50]
festival_sizes = [210]

for year in [2025]:
    festival_data_all = []
    for n_festival in festival_sizes:
        festival_data_all.extend(process_festival_lines(festival_lines, year, n_festival))

    artist_data_all = []
    for n_artist in artist_sizes:
        artist_data_all.extend(process_artist_lines(artist_lines, year, n_artist))

    df = pd.DataFrame(festival_data_all + artist_data_all)
    
    csv_name = f"all_tests/prompt_output_{year}.csv"
    df.to_csv(csv_name, index=False)


## 8B

In [ ]:
import pandas as pd
import time
import csv
import os

t2025 = pd.read_csv("all_tests/prompt_output_2025.csv")

for dataset, name in zip([t2025], ["t2025"]):

    file_path = "all_tests/prompt_output_Meta-Llama-3-8B"+str(name)+"_alltests.csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        
        if not file_exists:
            writer.writerow(dataset.columns.tolist() + ["answer"])
        
        for index, row in dataset.iterrows():
            question_text = row['prompt']
            
            response = client.chat.completions.create(
                model="llama-3.1-8b",
                messages=[{"role": "user", "content": question_text}],
                temperature=0,
            )
            
            answer_text = response.choices[0].message.content.strip()
            
            writer.writerow(list(row) + [answer_text])
            
            dataset.at[index, 'answer'] = answer_text
        


output_files = [
    "all_tests/prompt_output_Meta-Llama-3-8Bt2025_alltests.csv"
]

for file_path in output_files:
    df = pd.read_csv(file_path)
    
    df_selected = df[["id", "type","n_response","answer"]]
    
    new_file_path = file_path.replace(".csv", "_CM.csv")
    
    # Save to new CSV
    df_selected.to_csv(new_file_path, index=False)
    


## 70B

In [ ]:
import pandas as pd
import time
import csv
import os


t2025 = pd.read_csv("all_tests/prompt_output_2025.csv")

for dataset, name in zip([t2025], ["t2025"]):

    file_path = "all_tests/prompt_output_Meta-Llama-70B"+str(name)+"_alltests.csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        
        if not file_exists:
            writer.writerow(dataset.columns.tolist() + ["answer"])
        
        for index, row in dataset.iterrows():
            question_text = row['prompt']
            
            response = client.chat.completions.create(
                model="Meta-Llama-3.3-70B-Instruct-AWQ-INT4",
                messages=[{"role": "user", "content": question_text}],
                temperature=0,
            )
            
            answer_text = response.choices[0].message.content.strip()
            
            writer.writerow(list(row) + [answer_text])
            
            dataset.at[index, 'answer'] = answer_text
    

output_files = [
    "all_tests/prompt_output_Meta-Llama-70Bt2025_alltests.csv"
]

for file_path in output_files:
    df = pd.read_csv(file_path)
    
    df_selected = df[["id", "type", "n_response","answer"]]
    
    new_file_path = file_path.replace(".csv", "_CM.csv")
    
    df_selected.to_csv(new_file_path, index=False)
    


## Mixtral

In [ ]:
import pandas as pd
import time
import csv
import os

t2025 = pd.read_csv("all_tests/prompt_output_2025.csv")

for dataset, name in zip([t2025], ["t2025"]):

    file_path = "all_tests/prompt_output_mixtral-8x7b"+str(name)+"_alltests.csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        
        if not file_exists:
            writer.writerow(dataset.columns.tolist() + ["answer"])
        
        for index, row in dataset.iterrows():
            question_text = row['prompt']
            
            response = client.chat.completions.create(
                model="mixtral-8x7b",
                messages=[{"role": "user", "content": question_text}],
                temperature=0,
            )
            
            answer_text = response.choices[0].message.content.strip()
            
            writer.writerow(list(row) + [answer_text])
            
            dataset.at[index, 'answer'] = answer_text
        

output_files = [
    "all_tests/prompt_output_mixtral-8x7bt2025_alltests.csv"
]

for file_path in output_files:
    df = pd.read_csv(file_path)
    
    df_selected = df[["id", "type", "n_response","answer"]]
    
    new_file_path = file_path.replace(".csv", "_CM.csv")
    
    df_selected.to_csv(new_file_path, index=False)
    


# GPT

In [ ]:
import pandas as pd
import time
import csv
import os
from openai import OpenAI

client = OpenAI(api_key="key")


t2025 = pd.read_csv("all_tests/prompt_output_2025.csv")

for dataset, name in zip([t2025], ["t2025"]):

    file_path = "all_tests/prompt_output_GPT4omini"+str(name)+"_alltests.csv"
    file_exists = os.path.isfile(file_path)

    with open(file_path, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        
        if not file_exists:
            writer.writerow(dataset.columns.tolist() + ["answer"])
        
        for index, row in dataset.iterrows():
                question_text = row['prompt']
                response = client.responses.create(
                    model="gpt-4o-mini",
                    temperature=0,
                    input=question_text
                    )
                answer_text = response.output_text
                writer.writerow(list(row) + [answer_text])
                
                dataset.at[index, 'answer'] = answer_text
            
output_files = [
    "all_tests/prompt_output_GPT4ominit2025_alltests.csv"
]

for file_path in output_files:
    df = pd.read_csv(file_path)
    
    df_selected = df[["id", "type", "n_response","answer"]]
    
    new_file_path = file_path.replace(".csv", "_CM.csv")
    
    df_selected.to_csv(new_file_path, index=False)
    
